### Setup tools and libraries

In [1]:
# !pip install -q --upgrade openai-agents==0.2.2 "rich<14.0.0,>=13.7.0" python-dotenv requests pydantic
!pip install -q --upgrade openai-agents==0.2.2 

# Install the "helper" package from PyPI so it can be used in this notebook
!pip install helper

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for helper: filename=helper-2.5.0-py2.py3-none-any.whl size=19249 sha256=d05ea7d90ac47f9d6f32a00a4e5575c8558dcc526e8cd58742e06e7de6821736
  Stored in directory: /root/.cache/pip/wheels/ca/7c/ca/66e49ba7ea9c29fa8076ae549f74945f22d2468657cbdddab4
Successfully built helper


In [ ]:
import os, requests, asyncio

from openai import OpenAI
from IPython.display import display, Markdown
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, SQLiteSession
from pydantic import BaseModel
from typing_extensions import TypedDict
from datetime import datetime
from agents import RunResult  # Make sure RunResult is imported

# from helper import format_report_to_markdown
from agents import handoff, RunContextWrapper, CodeInterpreterTool, input_guardrail, GuardrailFunctionOutput, TResponseInputItem
from agents import Runner
from agents.extensions import handoff_filters
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

In [7]:
# Setup env. and Load API keys from .env file
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

if not openai_api_key:
    from getpass import getpass
    openai_api_key = getpass("OpenAI API key (will not be echoed): ")
    
# Ensure the agents/OpenAI client can read the key via the environment variable
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

if not tavily_api_key:
    from getpass import getpass
    tavily_api_key = getpass("Tavily API key (will not be echoed): ")

# Ensure the agents/OpenAI client can read the key via the environment variable
if tavily_api_key:
    os.environ["TAVILY_API_KEY"] = tavily_api_key

openai_client = OpenAI(api_key=openai_api_key)

print("✅ API keys loaded")
print(f"  OpenAI … {openai_api_key[:5]}***")
print(f"  Tavily … {tavily_api_key[:5]}***")

✅ API keys loaded
  OpenAI … sk-pr***
  Tavily … tvly-***


### Setup Model, Tavily Search Tool and Markdown class

In [8]:
# Define the models that we want to use
main_model = "gpt-6.1-terra"
small_model = "gpt-6.1-luna"

In [9]:
def print_markdown(txt: str):
    display(Markdown(txt))

In [ ]:
# Define the Tavily search function
class TavilyParams(TypedDict):
    query: str
    max_results: int

@function_tool
def tavily_search(params: TavilyParams) -> str:
    """Return a newline‑joined summary of Tavily results."""
    url = "https://api.tavily.com/search"
    payload = {
        "api_key": tavily_api_key,
        "query": params["query"],
        "max_results": params.get("max_results", 3),
    }
    resp = requests.post(url, json = payload, headers = {"Content-Type": "application/json"})
    if resp.status_code != 200:
        return f"Tavily error {resp.status_code}"
    items = resp.json().get("results", [])
    return "\n".join([f"- {itm['title']}: {itm['content']}" for itm in items]) or "No hits"

### Planner Agent - with Gaurdrail

In [ ]:
# We will define a Planner Agent that breaks down user research requests into 2-3 targeted web searches with reasons and queries

# This section defines a guardrail to prevent the agent from engaging in political topics. 
# It uses a dedicated LLM agent to classify the input as political or not, and triggers a tripwire if political content is detected.

# Define the output schema for the political topic guardrail.
# Whenever we classify something as political, we’ll store two fields:
# (1) is_political: a boolean flag (True/False).
# (2) reasoning: text explaining why.
# This makes the guardrail’s output predictable and structured.

class PoliticalTopicOutput(BaseModel):
    is_political: bool
    reasoning: str

# This builds an Agent (from the OpenAI Agents SDK) to detect political content.
# The instructions tell the model exactly what to do.
# The output_type enforces that the agent must return a PoliticalTopicOutput.

politics_guardrail_agent = Agent(
                            name = "Guardrail check",
                            instructions = """
                                    Check if the user is asking about political topics, politicians, elections, government policy, 
                                    or anything related to politics. 
                                    If so, set is_political to true and explain why in reasoning.""",
                            output_type = PoliticalTopicOutput
                        )

# @input_guardrail: decorator that marks the function 
# The function takes:
# ctx: a wrapper object holding context/state for the current run (things like conversation history, metadata, or session info).
# input: what the user typed in (either a string, or a structured list of TResponseInputItem).
# The function returns a GuardrailFunctionOutput: what did we find, and do we trip the guardrail?

@input_guardrail
async def politics_guardrail(
    ctx: RunContextWrapper[None], 
    agent: Agent, 
    input: str | list[TResponseInputItem]) -> GuardrailFunctionOutput:
    
    # Use the guardrail agent to classify the input.
    result = await Runner.run(politics_guardrail_agent, input, context = ctx.context)

    # output_info: stores the full structured output (is_political + reasoning).
    # tripwire_triggered: this is the key flag. If is_political == True, then the guardrail has been triggered.
    return GuardrailFunctionOutput(
        output_info = result.final_output,
        tripwire_triggered = result.final_output.is_political
    )

In [12]:
# Define a data model for one search plan item where each item contains:
# - reason: why this particular search is being done
# - query: the actual search string to run

class SearchPlanItem(BaseModel):
    reason: str
    query: str

# Define a data model for the complete search plan
# This contains a list of SearchPlanItems

class SearchPlan(BaseModel):
    searches: list[SearchPlanItem]

# Get today's date in YYYY-MM-DD format
date = datetime.now().strftime("%Y-%m-%d")

# Create an AI agent called "Planner"
planner_agent = Agent(
    name = "Planner",
    instructions = f"""
        Current date: {date} \n 
        
        Context: 
        You are a research planner agent tasked with designing a comprehensive research plan for a user request. 
        You have access to web search tools and should utilize the current date ({date}) when planning. 
        
        Instruction: 
        Break down the user's request into 3 distinct web searches, each with a clear reason and a specific query. 
        Ensure coverage of recent news, company fundamentals, risks, sentiment, and broader context. 
        
        Input: 
        The user's research request and the current date. 
        
        Output: 
        A list of search plan items, each with a 'reason' and a 'query', formatted as a JSON object matching the SearchPlan schema.""",
    model = main_model,
    output_type = SearchPlan,
    input_guardrails = [politics_guardrail] # THIS IS THE GUARDRAIL THAT PREVENTS POLITICAL TOPIC ASKS
)

In [13]:
# Let's test the AI Agent with the Guardrail!
q1 = "Why was Trump meeting with Putin last time?"
# q1 = "solid state battery companies"
run1 = await Runner.run(starting_agent = planner_agent, input = q1)
print_markdown(f"### 🤖 Agent’s Answer\n{run1.final_output}")

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire